# Lab 3 - Clustering | Unsupervised Learning

Here, we will cover fundamental concepts of unsupervised learning, mainly with K-means algorithm:

1. *Simple partitioning* algorithm applying K-means on synthetic dataset.
2. *Cluster quality* metrics to assess how homogeneous the clusters are.
3. *Experimental evaluation* to determine the best number of clusters for a specific algorithm.

---

```
Author:   Luis Quintero
Contact:  luis.quintero@dsv.su.se | luisqtr.com
```


---
First of all, we import the respective Python packages and initialize relevant variables

In [ ]:
# Numeric analysis
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt

# Packages used to create synthetic datasets
from sklearn.datasets import make_blobs, make_circles

In [ ]:
# Set the seed of the pseudo randomization to guarantee that results are reproducible between executions
RANDOM_SEED = 2025
np.random.seed(RANDOM_SEED)

# 1. Simple partitioning with K-means

To explain the clustering algorithm by partitions, we will execute the following steps:
- First, we create a synthetic dataset that has some predefined clusters.
- Then, we create a function that implements the K-means algorithm manually (without using the predefined libraries)
- We apply our clustering algorithm step by step on the synthetic dataset
- Use the clustering algorithm from sklearn and see the results compared to our method
- Apply clustering in a second synthetic dataset (circle-shaped) and analyze the clustering results

### Creating a simple synthetic dataset

First, we create a synthetic dataset that is easy to cluster. It serves the educational purpose of showing what K-means algorithm is doing behind the scenes. Below, we create a synthetic dataset that contains 40 observations and each observation has 2 features.

*Theoretical clarification:*

Note that the function `make_blobs()` ([read docs here](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html?highlight=make_blobs#sklearn.datasets.make_blobs)) produces synthetic data in the shape of blobs ($X$) but it also returns the **labels** ($y$) indicating to which group each sample belongs.

The fact that it has labels might cause confusion now because we defined that **clustering algorithms** like K-means do not utilize data labels to group data samples in $X$. However, in this case, we will use the true labels to *evaluate* the result of the clustering algorithm and not to *train* the algorithm.. This distinction will be clearer in future labs when we work with **clasification algorithms**.

In [ ]:
# Feature and real class labels
data_X, data_y_cluster = make_blobs(n_samples=40, n_features=2,
                                        centers=4,  # There will be 4 true groups
                                        random_state=RANDOM_SEED)

The previous function returns **numpy arrays**, as required for most functions in scikit-learn and matplotlib.

We can easily convert them to a pandas dataframe.

In [ ]:
data = pd.DataFrame({
                        "x1": data_X[:,0],
                        "x2": data_X[:,1],
                        "y_real": data_y_cluster
                        })

# Assign data type to group membership as a 'nominal category'
data["y_real"] = data["y_real"].astype("category")

# Show first values from the data
data.head()

In [ ]:
data.plot.scatter(x="x1",y="x2", s=30, title="Synthetic blobs dataset")

The plot above shows only the data samples $X$, which is the only information that we need for the clustering algorithm. However, we can also visualize their **assigned real cluster** in $y$ with different colors, and we will just use this information later on to assess our cluster quality, not to generate the clusters themselves.

In [ ]:
data["y_real"].values

### Normalization and Standardization (*Very important for clustering*)

Before we continue, note that the dataset contains different ranges in the x-axis and the y-axis (variable $x_1$ spans along $[-8,6]$, whereas $x_2$ is between $[-11,6]$).

This discrepancy will heavily affect the clustering algorithm, therefore it is necessary to **normalize** (a.k.a. min-max scaler) the values to force a range between $[0,1]$.

$$x' = \frac{(x - x_{min})}{(x_{max} - x_{min})}$$

*OBS:* An alternative preprocessing approach is to transform to **standardize** (a.k.a. Z-scores), which forces the ranges to values with mean $\mu=0$ and standard deviation $\sigma=1$.

$$x' = \frac{(x - \mu_x)}{\sigma_x}$$

Here, we will *normalize* the variables below, but the syntax of the code below is explained in detail later in the lab.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
data_X = min_max_scaler.fit_transform(data_X)

print(f"Minimum value in data is: {np.min(data_X)}")
print(f"Maximum value in data is: {np.max(data_X)}")

In [ ]:
# A plot showing the normalized data
# using the package matplotlib

# Scatter plot to visualize the main data in `X` with the colors in `y`
plt.scatter(x = data_X[:,0], y = data_X[:,1], c=data_y_cluster, s=30)

# Format the plot
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Synthetic blobs dataset | Real clusters")
plt.grid(True)
plt.show()

Before starting the clustering algorithms, let's create a function to visualize the dataset and the clusters.

There is no need to understand the function below in full detail, but you will see later in the code that it is handy to have custom functions in the data analysis pipeline that simplifies the understanding of the clustering algorithm.

In [ ]:
def visualize_clustering_algorithm(X, y, centers=None, title=""):
    """
    Input:
        X: (np.array[N,2]) - The dataset to be clustered
        y: (np.array[N,1]) - Corresponding cluster memberships
        centers: (np.array[K,2]) - [optional parameter]
                    Shows where the current cluster centers are.
    Returns:
        A plot with the dataset and the centroids of the clusters
    """
    # Scatter plot to visualize the main data in `X` with the colors in `y`
    plt.scatter(x = X[:,0], y = X[:,1], c=y, s=30, label="Dataset")

    # If the user passes an array of `centers`,
    # then draw another scatterplot on top and show a legend
    if(centers is not None):
        plt.scatter(x = centers[:,0], y = centers[:,1],
                    c=np.arange(centers.shape[0]),
                    s=200, marker="x",
                    label="Cluster centers")
        plt.legend()

    # Format the plot
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.grid(True)
    return plt.show()

In [ ]:
visualize_clustering_algorithm(data_X, data_y_cluster, title="Real clusters")

### Implement K-means manually

Here, we will create a function that runs the K-means algorithm manually (without using packages but following the mathematical algorithm).

The goal is to understand better how to create and utilize functions. All the programming libraries we `import` at the beginning are ultimately a compilation of functions that someone else wrote to facilitate the workflow for our analysis.

When exploring the notebook, you can just execute the cells below and see what the outputs are, no need to understand the internal code in detail at the beginning. However, when doing the homework, it might be useful to analyze the algorithm in detail.

**OBS❕❕:** The function below **only** implements 1 iteration of the K-means algorithm... Not all iterations of K-means.

**K-Means algorithm**

The algorithm clusters a set of data $X$ based on the *similarity* between their points.

The K-means algorithm requires as input:

- A 2D-array with the numerical data $X$ of size $(N,M)$.
- The number of clusters $k$.

The algorithm returns:

- An array of size $(N,1)$ indicating the cluster membership of each sample ($0,1,...,k-1$)
- An array with the center coordinates of each cluster $(k,M)$.

In [ ]:
def k_means_manual_1_iteration(X,k):
    """
    NOTE: This function **only** performs 1 iteration of
          the K-means algorithm. Not until convergence.
    Input:
        X: (np.array[N,M]) - The dataset to be clustered
        k: (int) - Number of desired clusters
    Returns:
        centers: (np.array[k,M]) - Coordinates of the cluster centers
        labels: (np.array[N,1]) - Cluster memberships as found by K-means
    """
    ## Creating variables that are specifically for the function
    N,M = X.shape       # Read the size of the input data
    num_clusters = k    # This values is provided by the user as a parameter

    print(f"Input dataset has size [N,M]=[{N},{M}]")
    print(f"Number of clusters to create k={num_clusters}")

    ## Intermediate variables where we will store the result
    ## Initialized with zeros because we will put the values later
    centers = np.zeros((num_clusters,M))
    labels = np.zeros(N)

    ## NOTE: Review the theoretical steps of the K-means algorithms
    ## from the lectures if you don't remember the following steps...

    # 1. Randomly initialize the cluster centers

    # Here we just take the last k samples of the dataset as centroids
    # This is for the sake of simplicity, but other algorithms like K-means++
    # have better approaches to choose the initialization points
    centers = X[-num_clusters:,:].copy()

    visualize_clustering_algorithm(X,labels,centers,title="Initialization")

    # 2. Calculate the distance from each sample to each center

    # Let's create a temporary array that helps us store distances from each sample
    # to each of the cluster centers. Therefore the size is `N` rows by `num_clusters` columns
    distance_to_centers = np.zeros((N,num_clusters))

    # Iterate over the samples to populate the array
    for i in range(N):  # Iterate over each data sample
        sample = X[i,:] # Select one row in the dataset

        # Iterate over each cluster center
        for j in range(num_clusters):
            center = centers[j,:] # Select one cluster center

            # 2.1) Calculate the distance to each center.
            ### NOTE: Here it is where we define our **similarity metric**
            # In this case we use Euclidean distance (L2-norm).
            distance_to_centers[i, j] = np.linalg.norm(sample - center)

        # 2.2) Assign the label corresponding to the closest center (minimum distance)
        index_of_closest_center = np.argmin(distance_to_centers[i,:])
        labels[i] = index_of_closest_center

    visualize_clustering_algorithm(X,labels, centers, title="Assigning a group to each data point")

    # When the iteration has finished for all the samples, calculate new cluster centers
    for j in range(num_clusters):
        # Filter samples belonging to the cluster
        samples_in_cluster = X[ labels == j ]
        centers[j, :] = np.mean(samples_in_cluster,axis=0)

    visualize_clustering_algorithm(X,labels,centers,title="Cluster groups after recalculating centers | 1 iteration")

    # Return from the function the corresponding values after applying K-means
    return centers, labels


In [ ]:
print(f"Original dataset size: {data_X.shape} has these last four observations:\n{data_X[-4:,:]}")

estimated_centers, estimated_clusters = k_means_manual_1_iteration(data_X, 4)

Note that when we called the function `k_means_manual_1_iteration()` we performed **only one iteration of the K-means algorithm**, not until it converges.

Also, when passing the parameter `k` that indicates the number of desired clusters, we passed `4` clusters because we knew in advance that the original dataset was generated with 4 blobs. But this may be considered as if we are **cheating** 😱😱❗❗, because sometimes in real-life tasks we actually want to know *how many clusters we can create from the data*. So far let's assume we want to create 4 clusters, later we will show how to determine experimentally the best number of clusters `k` based on evaluation metrics.

How do the clusters above compare with the real clusters?

First, let's calculate the cluster centroids $(c_1,c_2)$ from the real cluster membership:

In [ ]:
def calculate_centers_from_labels(X, labels):
    """
    Input:
        X: (np.array[N,M]) - The dataset to be clustered
        labels: (np.array[N,1]) - Cluster memberships
    Returns:
        centers: (np.array[k,M]) - Coordinates of the cluster centers
    """
    N,M = X.shape
    num_clusters = np.unique(labels).size
    ctrs = np.zeros((num_clusters,M))
    for i in range(num_clusters):
        data_in_cluster = X[ labels==i ]
        # The centers correspond to the mean value of the data belonging to the cluster i
        ctrs[i,:] = np.mean(data_in_cluster,axis=0)
    return ctrs

In [ ]:
# How does it compare with the real clusters?
data_centers_real = calculate_centers_from_labels(data_X, data_y_cluster)
print(f"Based on the labels, there are clusters with centers in: {data_centers_real}")

In [ ]:
# Plot the original clusters in the dataset
visualize_clustering_algorithm(data_X, data_y_cluster, centers=data_centers_real, title="Real clusters")

As we see in the images above, we finished **1 iteration** of the K-means algorithm and the clusters start getting grouped based on how close the data samples are to each other. However, we are far from the real clusters, and it is because we have only done the iteration **ONCE**.

Below, there is a modified version where we add a `while` loop that iterates the same algorithm until a stopping condition is satisfied (a.k.a. *the algorithm converges*). In this case, the stopping condition is the cluster centers have not changed between two subsequent iterations or that the algorithm iterates more than 100 times.

In [ ]:
def k_means_manual_until_convergence(X,k):
    """
    Input:
        X: (np.array[N,M]) - The dataset to be clustered
        k: (int) - Number of clusters that we want to create
    Returns:
        centers: (np.array[k,M]) - Coordinates of the cluster centers
        labels: (np.array[N,1]) - Cluster memberships as found by K-means
    """
    ## Creating variables that are specifically for the function
    N,M = X.shape       # Read the size of the input data
    num_clusters = k    # This values is provided by the user as a parameter

    print(f"Input dataset has size [N,M]=[{N},{M}]")
    print(f"Number of clusters to create k={num_clusters}")

    ## Intermediate variables where we will store the result
    ## Initialized with zeros because we will put the values later
    centers = np.zeros((num_clusters,M))
    labels = np.zeros(N)

    ## NOTE: Review the theoretical steps of the K-means algorithms
    ## from the lectures if you don't remember the following steps...

    # 1. Randomly choose a starting point for the cluster centers

    centers = X[-num_clusters:,:].copy()   # Here we just take the last k samples of the dataset as centroids

    visualize_clustering_algorithm(X,labels,centers,title="Initialization")

    # 2. Calculate the distance from each sample to each center

    # Let's create a temporary array that helps us store distances from each sample
    # to each of the cluster centers. Therefore the size is N rows by k columns
    distance_to_centers = np.zeros((N,num_clusters))

    """
    Note that the only difference with respect to the function `k_means_manual_1_iteration()`
    is that we are iterating the same process multiple times
    """
    previous_centers = np.zeros((num_clusters,M))   # Variable to keep track when the centers changed
    counter_iterations = 0          # We will also keep track of how many iterations are performed

    # The stopping condition between parentheses define the convergence criteria of the algorithm
    while ( (not np.array_equal(previous_centers,centers)) and counter_iterations < 100):

        """
        All the steps below are the same than in the function `k_means_manual_1_iteration()`
        """

        # Iterate over the samples to populate the array
        for i in range(N):  # Iterate over each data sample
            sample = X[i,:]
            for j in range(num_clusters):   # Iterate over each cluster center
                center = centers[j,:]

                # 2.1) Calculate the distance to each center.
                ### NOTE: Here it is where we define our **similarity metric**
                # In this case we use Euclidean distance (L2-norm).
                distance_to_centers[i, j] = np.linalg.norm(sample - center)

            # 2.2) Assign the label corresponding to the closest center (minimum distance)
            index_of_closest_center = np.argmin(distance_to_centers[i,:])
            labels[i] = index_of_closest_center

        """
        NEW!!! Here we are storing the previous cluster centers before they are recalculated
        """
        previous_centers = centers.copy()
        counter_iterations = counter_iterations+1

        # When the iteration has finished for all the samples, calculate new cluster centers
        for j in range(num_clusters):
            # Filter samples belonging to the cluster
            samples_in_cluster = X[ labels == j ]
            centers[j, :] = np.mean(samples_in_cluster,axis=0)

        # Now, we visualize the results just at the end of the iterations
        visualize_clustering_algorithm(X,labels,centers,title=f"Clusters groups after {counter_iterations} iterations")

    # Return from the function the corresponding values after applying K-means
    return centers, labels


### Apply our implementation of K-means to the dataset

In [ ]:
print(f"Size: {data_X.shape} and last objects:\n{data_X[-4:,:]}")

# Run the k-means until it converges
estimated_centers, estimated_clusters = k_means_manual_until_convergence(data_X, 4)

Let's simplify the visualization with a function that puts next to each other the real clusters and the estimated clusters.

You don't need to understand the details of this implementation, but you need to understand how the function is used because you will use it for your homework.

In [ ]:
## Simpler visualization of the resulting clusters

def visualize_and_compare_clusters(X, real_labels, real_centers, estimated_labels, estimated_centers):
    """
    Input:
        X: (np.array[N,2]) - The dataset to visualize (only 2 features)
        real_labels: (np.array[N,1]) - Real cluster membership
        real_centers: (np.array[K,2]) - Real cluster centers
        estimated_labels: (np.array[N,1]) - Estimated cluster membership from algorithm
        estimated_labels: (np.array[K,2]) - Estimated cluster centers
    Returns:
        A plot with two axes showing the real clusters and the predicted clusters
    """

    fig, axes = plt.subplots(1, 2, figsize=(10,5))

    # First plot contains real cluster membership
    ax = axes[0]
    ax.scatter(x = X[:,0], y = X[:,1], c=real_labels, s=30, label= "Data")
    ax.scatter(x = real_centers[:,0], y = real_centers[:,1],
            c=np.arange(real_centers.shape[0]), s=200, marker="x", label="Centers")
    ax.set(xlabel="x1",ylabel="x2",title="Real clusters")
    ax.grid(True); ax.legend()

    # Second plot contains estimated cluster membership
    ax = axes[1]
    ax.scatter(x = X[:,0], y = X[:,1], c=estimated_labels, s=30, label="Data")
    ax.scatter(x = estimated_centers[:,0], y = estimated_centers[:,1],
            c=np.arange(estimated_centers.shape[0]), s=200, marker="x", label="Centers")
    ax.set(xlabel="x1",ylabel="x2",title="Estimated clusters")
    ax.grid(True); ax.legend()

    return plt.show()


In [ ]:
visualize_and_compare_clusters(data_X,
                               data_y_cluster, data_centers_real,
                               estimated_clusters, estimated_centers)

## `NOTE:` ***Visual analysis of the clusters***

As we mentioned in previous Labs, the role of data science goes beyond having the models that work or not. There needs to be a critical appraisal on the results that we are seeing.

Reflect on the following questions. No need to provide answers yet, but **`you will reflect on the similar questions during your homework`**:

1. Are the estimated clusters equal to the real clusters?
2. Are the estimated centers close to the real centers?

Remember from the lectures that K-means has some caveats. For example, "*the choice of initial points can have large influence in the result*".

[Read here](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_assumptions.html#sphx-glr-auto-examples-cluster-plot-kmeans-assumptions-py) regarding the possible problems with K-means and possible solutions.

1. Do you see any problem in your cluster caused by bad initialization? Why or why not?
2. Would the resulting clusters be eventually useful or do you need to try new hyperparameters or more algorithms? Why? Why not?

# Apply K-means from Python library

The package `scikit-learn` offers all the functionalities that we implemented above and many other algorithms. However, implementing the algorithms manually with `numpy` provides a deeper understanding on the mathematical principles and more flexibility to handle the data as required.

From now on, **we will continue using sklearn**, and the example above should serve as a guide if you need to implement a more specific analysis that is not provided by existing libraries. For instance, as we need to do in [data science research](https://datascience.dsv.su.se/) or in a thesis project in the field.

In [ ]:
# Importing the necessary functions from sklearn
from sklearn.cluster import KMeans

All functions from `sklearn` have a very simple workflow:
1. First, we need to initialize an object with the algorithm that we want, in this case it is the `KMeans()` function, which can be customized with multiple parameters ([see docs here](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)).
2. Then, we need to apply the algorithm in our **training data** by running the function `.fit()` with the corresponding parameters, which creates/trains/builds the **machine learning model**.
3. Lastly, we generate new labels on **unseen test data** by executing the function `.predict()` with the corresponding parameters.
4. A common function is to execute `.fit_predict()` as a shortcut for both actions.

In [ ]:
# 1. Initialize a KMeans object that looks for 4 clusters in the data
kmeans = KMeans(n_clusters=4, random_state=RANDOM_SEED,
                init='random', n_init='auto', max_iter=100)

# 2. Apply the object on our dataset
kmeans.fit(data_X)

# 3. Extract the estimated labels from the data.
# We would run the function `.predict()` only for unseen data, but
# here we just want to see the actual labels of the original training set
# where the model was trained)
data_y_kmeans = kmeans.labels_

print(f"The estimated cluster membership is:\n{data_y_kmeans}")

In [ ]:
# Extract the cluster centers according to KMeans
cluster_centers_kmeans = kmeans.cluster_centers_
# Apply the inverse transformation to get the values in the original space
centers_numpy = min_max_scaler.inverse_transform(cluster_centers_kmeans)
# Visualize the centroids as pandas dataframe, not numpy array.
cluster_centers_df = pd.DataFrame(data=centers_numpy,
                                  columns=["x1","x2"])
cluster_centers_df

In [ ]:
# Plot the real and the estimated with K-means
visualize_and_compare_clusters(data_X,
                               data_y_cluster, data_centers_real,
                               data_y_kmeans, cluster_centers_kmeans)


The KMeans algorithm from sklearn should be able to estimate the same clusters than the real clusters (*regardless the color of each cluster*). The reason is because the dataset has very well defined groups that can be separated linearly.


# Cluster Evaluation

So far, we are assessing the quality of the clusters **visually**, taking the data from the real clusters to analyze whether our clustering algorithm. When we use the original labels to assess the clusters, these are called **external metrics**. However, as mentioned earlier, we don't always have the real labels at hand, and there are other **internal metrics**, like *Silhouette score*, that assess the quality of the clusters based on the homogeneity of each cluster.

The extensive example of clustering performance evaluation is available in [scikit-learn documentation](https://scikit-learn.org/stable/modules/clustering.html#clustering-evaluation).

Next, we show how to use scikit-learn to assess the cluster's quality in the latest dataset of concentric circles.

In [ ]:
# Import the codes containing the evaluation metrics
from sklearn import metrics

In [ ]:
# Before we created some variables with estimated clusters per algorithm.
# Let's recall our variables.
print(f"The real cluster labels are in `data_y_cluster`: {data_y_cluster.shape}")
print(f"The cluster labels estimated by KMeans are in `estimated_clusters_circles_kmeans`: {data_y_kmeans.shape}")

In [ ]:
# Let's calculate the numerical comparison per algorithm.

# External metrics, using the real cluster memberships
eval_rand_index_real = metrics.adjusted_rand_score(labels_true = data_y_cluster,
                                                labels_pred = data_y_kmeans)
print("The adjusted rand index for the Real Clusters is:", eval_rand_index_real)

eval_rand_index_kmeans = metrics.adjusted_rand_score(labels_true = data_y_cluster,
                                                labels_pred = data_y_kmeans)
print("The adjusted rand index for the KMeans Clustering algorithm is:", eval_rand_index_kmeans)

# Internal metrics without knowledge of real cluster membership labels

eval_silhouette_score_kmeans = metrics.silhouette_score(data_X, data_y_kmeans, metric="euclidean")
print("The silhouette score for the KMeans Clustering algorithm is:", eval_silhouette_score_kmeans)


External libraries can help you visualize silhouette score to facilitate the interpretation. For example, [Yellowbrick](https://www.scikit-yb.org/en/latest/api/cluster/silhouette.html)

# Experimental Evaluation to infer number of clusters

During the whole lab, we have been defining the `number of clusters` according to our pre-existing knowledge of the data. Now, let's use the numerical internal cluster evaluation to define the best number of clusters.

The example below just creates a new instance of the agglomerative clustering algorithm varying the number of desired clusters, and calculates the silhouette score for each iteration. We know a priori that the datset has 2 clusters but, *does the experimental evaluation suggest the same number of clusters?*

In [ ]:
# Let's try from 2 until 8 clusters, and compare which one is better

clusters_to_evaluate = np.arange(2,9) # Generates a sequence from 2 until 8.
print(f"Testing num_clusters: {clusters_to_evaluate}")

# Store the results in a dataframe, initialize empty array
results = pd.DataFrame({"k": [],
                        "silhouette_score": []
                        })

# Iterate
for k in clusters_to_evaluate:
    # Note that the parameter `n_clusters` is the variable `k`
    kmeans = KMeans(n_clusters = k, random_state=RANDOM_SEED, init='random', n_init='auto', max_iter=100)
    # Generate cluster labels for the data
    predicted_clusters = kmeans.fit_predict(data_X)
    # Calculate silhouette score
    sil_score = metrics.silhouette_score(data_X, predicted_clusters)

    # Add the results to the pandas dataframe
    this_result = pd.DataFrame({"k": [k],
                                "silhouette_score": [sil_score]})

    results = pd.concat([results, this_result], ignore_index=True)


results["k"] = results["k"].astype(np.int16)

# Print results
print("The silhouette score for each number of clusters is presented below:")
print(results)


In [ ]:
results.plot.bar(x="k",y="silhouette_score", title="Silhouette score per number of clusters")

### Question: Which number of cluster yields better clustering performance according to the Silhouette Score?

# Example 2 : Concentric circles
Let's try to apply K-means on a harder dataset. The following dataset contains two groups of data (clusters, classes) in concentric circles.

Note that the dataset below does not need normalization. `Question for you: Why?`

In [ ]:
# Create the other dummy dataset
data_circles_X, data_circles_real_cluster = make_circles(n_samples = 500, factor=0.5, noise=0.05, random_state=RANDOM_SEED)

# Calculate from now what would be the real centers of the clusters
data_circles_real_centers = calculate_centers_from_labels(data_circles_X, data_circles_real_cluster)

visualize_clustering_algorithm(data_circles_X, data_circles_real_cluster, centers=None, title="Dataset Circles")

In [ ]:
# Apply K-Means and create labels for the data.
#  - Assume that we know already the perfect number of clusters = 2
kmeans = KMeans(n_clusters=2, init='random', n_init='auto')
kmeans.fit(data_circles_X)
estimated_clusters_circles_kmeans = kmeans.labels_

# Calculate the coordinates of the real and estimated centers
estimated_centers_circles_kmeans = calculate_centers_from_labels(data_circles_X, estimated_clusters_circles_kmeans)

# Visualize
visualize_and_compare_clusters(data_circles_X,
                                data_circles_real_cluster, data_circles_real_centers, # Calculated some cells above
                                estimated_clusters_circles_kmeans, estimated_centers_circles_kmeans)


`Let's reflect on the results again!`

### ***Visual analysis of the clusters***

1. Are the estimated clusters equal to the real clusters?
2. Are the estimated centers close to the real centers?
3. Do you see any problem in your clustering approach? Why or why not?
4. Would the resulting clusters be eventually useful or do you need to try new hyperparameters or more algorithms? Why? Why not?

Most likely the clustering results where not close to the cluster generated by the algorithm.

K-Means has a few limitations. For example, it assumes that clusters are [convex and isotropic](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_assumptions.html), which is not always the case. It also assumes that K (the number of clusters) is known in advance, which is often not the case in real-world applications. Despite these limitations, K-Means is a powerful tool and forms the basis for many other clustering algorithms.

Let's try another clustering algorithm and see the results.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agg_clust_single = AgglomerativeClustering(n_clusters=2, linkage="single")

# As a shortcut, we are executing `fit_predict` to immediately get the labels on the dataset
estimated_clusters_circles_agg_s = agg_clust_single.fit_predict(data_circles_X)

# Calculate the coordinates of the real and estimated centers
estimated_centers_circles_agg_s = calculate_centers_from_labels(data_circles_X, estimated_clusters_circles_agg_s)

# Visualize
visualize_and_compare_clusters(data_circles_X,
                                data_circles_real_cluster, data_circles_real_centers,
                                estimated_clusters_circles_agg_s, estimated_centers_circles_agg_s)


The estimated clusters above should look much better than with KMeans.

As usual, scikit-learn contains extensive documentation on how to use the library for the technical implementation. For example, see the explanation of K-means++ in [this link](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_plusplus.html?highlight=make_blobs). These lab notebooks mostly intend to share insights about the non-technical aspects of data science, such as pondering when and why to use a specific method.

---

# Activities - [Not graded]

1. Calculate the Rand Index and Silhouette score of both  KMeans and Agglomerative clustering in the example of the concentric circles. Interpret the number and answer these questions:

  * What is the range of values for the metric *Rand Index*? Did KMeans or
Agglomerative Clustering performed better?
  * What is the range of values for the metric *Silhouette score* (*[hint](https://scikit-learn.org/stable/modules/clustering.html#silhouette-coefficient)*)? Did KMeans or Agglomerative Clustering performed better?
  * Would any of these machine learning models be sufficient to cluster the data?
  * Would it be necessary to try more hyperparameters or different algorithms? Why? Why not?
  * Why does KMeans yield higher silhouette score than Agg. Clustering if the visual inspection showed that it was not good at clustering?


2. Load and process the the `bank.csv` file from Lab 2 to answer the analytical question that we left unanswered: *How many groups of similar customers can be created from the bank dataset?*

  * Load the dataset and select only the two numerical columns (age, balance)
  * Perform an experimental evaluation to find the best number of clusters K for the KMeans algorithm. Take as a reference the code in this lab.
  * Run again the clustering algorithm with the preferred number of clusters K and visualize the cluster in a figure.
  * Conduct a visual analysis of the clusters: Do you see any problem in your cluster caused by bad initialization? Why or why not? Would the resulting clusters be eventually useful or do you need to try new hyperparameters or more algorithms? Why? Why not?
  * Calculate Silhouette score and do a numerical analysis of the results: Is KMeans sufficient to cluster the data effectively? Why? Why not? Would it be necessary to try more algorithms or the data is impossible to cluster using only age and balance?
  * Could you calculate Rand Index? Why? Why not